In [3]:
import torch
import numpy as np

# 체크포인트 경로
#ckpt1_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Few/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-BASELINE3/42/Embed:carte_Edge:mlp_A:gat_v1_S:42_20251204_210327.pt"
#ckpt2_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Few/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-BASELINE5/42/Embed:carte_Edge:mlp_A:gat_v1_S:42_20251204_210959.pt"
#ckpt1_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/hungarian/Few/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_struct_hidden_dim-192_fgw_alpha-1_alpha-0.9_vq_beta-0.3_kl_gamma-2.0_tau-0.5_target_data-heart_entropic_reg-0.01_description-None/42/20251217_225625/Embed:carte_Edge:mlp_A:gat_v1_S:42_20251218_000305.pt"
#ckpt1_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/hungarian/Few/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_struct_hidden_dim-192_fgw_alpha-1_alpha-0.9_vq_beta-0.3_kl_gamma-2.0_tau-0.5_target_data-heart_entropic_reg-0.01_description-None/42/20251217_225625/Embed:carte_Edge:mlp_A:gat_v1_S:42_20251217_225625.pt" 
#ckpt2_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/hungarian/Few/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_struct_hidden_dim-192_fgw_alpha-1_alpha-0.9_vq_beta-0.3_kl_gamma-2.0_tau-0.5_target_data-heart_entropic_reg-0.01_description-None/42/20251217_225625/Embed:carte_Edge:mlp_A:gat_v1_S:42_20251218_000306.pt"
ckpt1_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/hungarian/Few/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_struct_hidden_dim-192_fgw_alpha-1_alpha-0.9_vq_beta-0.3_kl_gamma-2.0_tau-0.5_target_data-heart_entropic_reg-0.01_description-None/42/20251217_225625/Embed:carte_Edge:mlp_A:gat_v1_S:42_20251218_000305.pt"
ckpt2_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/hungarian/Few/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_struct_hidden_dim-192_fgw_alpha-1_alpha-0.9_vq_beta-0.3_kl_gamma-2.0_tau-0.5_target_data-heart_entropic_reg-0.01_description-None/42/20251217_225625/Embed:carte_Edge:mlp_A:gat_v1_S:42_20251218_000308.pt"

def compare_tensors(t1, t2, name=""):
    """Tensor 비교 함수"""
    if t1.shape != t2.shape:
        print(f"  {name}: ❌ Different shapes: {t1.shape} vs {t2.shape}")
        return False
    
    is_equal = torch.allclose(t1, t2, rtol=1e-5, atol=1e-8)
    if not is_equal:
        max_diff = torch.abs(t1 - t2).max().item()
        mean_diff = torch.abs(t1 - t2).mean().item()
        print(f"  {name}: ❌ Values differ")
        print(f"     Shape: {t1.shape}")
        print(f"     Max difference: {max_diff:.2e}")
        print(f"     Mean difference: {mean_diff:.2e}")
        return False
    return True

def compare_dicts(d1, d2, prefix=""):
    """Dictionary 재귀 비교 함수"""
    keys1 = set(d1.keys())
    keys2 = set(d2.keys())
    
    all_same = True
    
    if keys1 != keys2:
        print(f"  {prefix}: ❌ Different keys")
        print(f"    Only in d1: {keys1 - keys2}")
        print(f"    Only in d2: {keys2 - keys1}")
        return False
    
    for key in keys1:
        val1 = d1[key]
        val2 = d2[key]
        full_key = f"{prefix}.{key}" if prefix else key
        
        if isinstance(val1, torch.Tensor):
            if not compare_tensors(val1, val2, full_key):
                all_same = False
        elif isinstance(val1, dict):
            if not compare_dicts(val1, val2, full_key):
                all_same = False
        else:
            if val1 != val2:
                print(f"  {full_key}: ❌ Values differ: {val1} vs {val2}")
                all_same = False
    
    return all_same

# 체크포인트 로드
print("Loading checkpoints...")
ckpt1 = torch.load(ckpt1_path, map_location='cpu')
ckpt2 = torch.load(ckpt2_path, map_location='cpu')

print("\n" + "="*80)
print("CHECKPOINT COMPARISON")
print("="*80)

# 1. 키 비교
keys1 = set(ckpt1.keys())
keys2 = set(ckpt2.keys())

print("\n[1] Key Comparison:")
print(f"  Checkpoint 1 keys: {len(keys1)}")
print(f"  Checkpoint 2 keys: {len(keys2)}")
print(f"  Common keys: {len(keys1 & keys2)}")
if keys1 - keys2:
    print(f"  Only in ckpt1: {keys1 - keys2}")
if keys2 - keys1:
    print(f"  Only in ckpt2: {keys2 - keys1}")

# 2. 공통 키에 대해 값 비교
print("\n[2] Value Comparison (for common keys):")
print("-"*80)

all_same = True
for key in sorted(keys1 & keys2):
    val1 = ckpt1[key]
    val2 = ckpt2[key]
    
    # 타입이 다른 경우
    if type(val1) != type(val2):
        print(f"\n{key}:")
        print(f"  ❌ Different types: {type(val1)} vs {type(val2)}")
        all_same = False
        continue
    
    # Tensor인 경우
    if isinstance(val1, torch.Tensor):
        # shape 비교
        if val1.shape != val2.shape:
            print(f"\n{key}:")
            print(f"  ❌ Different shapes: {val1.shape} vs {val2.shape}")
            all_same = False
            continue
        
        # 값 비교
        is_equal = torch.allclose(val1, val2, rtol=1e-5, atol=1e-8)
        
        if not is_equal:
            max_diff = torch.abs(val1 - val2).max().item()
            mean_diff = torch.abs(val1 - val2).mean().item()
            print(f"\n{key}:")
            print(f"  ❌ Values differ")
            print(f"     Shape: {val1.shape}")
            print(f"     Max difference: {max_diff:.2e}")
            print(f"     Mean difference: {mean_diff:.2e}")
            print(f"     Ckpt1 - mean: {val1.mean().item():.6f}, std: {val1.std().item():.6f}")
            print(f"     Ckpt2 - mean: {val2.mean().item():.6f}, std: {val2.std().item():.6f}")
            all_same = False
        else:
            print(f"{key}: ✅ Identical (shape: {val1.shape})")
    
    # Dictionary인 경우
    elif isinstance(val1, dict):
        print(f"\n{key}: (dict with {len(val1)} keys)")
        if not compare_dicts(val1, val2, key):
            all_same = False
        else:
            print(f"  ✅ All values identical")
    
    # 기타 타입
    else:
        try:
            if val1 != val2:
                print(f"\n{key}:")
                print(f"  ❌ Values differ: {val1} vs {val2}")
                all_same = False
            else:
                print(f"{key}: ✅ Identical ({type(val1).__name__})")
        except:
            print(f"\n{key}: ⚠️  Could not compare ({type(val1).__name__})")

print("\n" + "="*80)
if all_same and keys1 == keys2:
    print("✅ RESULT: Checkpoints are IDENTICAL")
else:
    print("❌ RESULT: Checkpoints are DIFFERENT")
print("="*80)

Loading checkpoints...

CHECKPOINT COMPARISON

[1] Key Comparison:
  Checkpoint 1 keys: 6
  Checkpoint 2 keys: 6
  Common keys: 6

[2] Value Comparison (for common keys):
--------------------------------------------------------------------------------
args: ✅ Identical (Namespace)
epoch: ✅ Identical (int)

model_state_dict: (dict with 158 keys)
  ✅ All values identical
threshold: ✅ Identical (float32)
val_auc: ✅ Identical (float)
val_auprc: ✅ Identical (float)

✅ RESULT: Checkpoints are IDENTICAL


In [1]:
import torch

# 파일 경로 설정
ckpt_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-BASELINE5/42/best.pt"

print(f"Loading checkpoint from: {ckpt_path} ...\n")

# CPU로 로드 (GPU 메모리 부족 방지 및 호환성 위함)
try:
    checkpoint = torch.load(ckpt_path, map_location='cpu')
    
    # 1. 최상위 키 확인 (args, epoch, model_state_dict 등이 무엇이 있는지)
    print("=== [1] Top-level Keys ===")
    print(checkpoint.keys())
    print("\n")

    # 2. 모델 파라미터(state_dict) 차원 확인
    print("=== [2] Model Parameter Dimensions ===")
    
    # 보통 'model_state_dict' 혹은 'state_dict' 키에 저장됩니다.
    # 만약 키가 없다면 checkpoint 자체가 state_dict일 수도 있습니다.
    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    elif 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint # 통째로 저장된 경우

    # 각 레이어의 이름과 쉐입 출력
    for key, tensor in state_dict.items():
        if torch.is_tensor(tensor):
            print(f"{key:<50} | Shape: {tuple(tensor.shape)}")
        else:
            print(f"{key:<50} | Value: {tensor} (Not a tensor)")

    # 3. 저장된 Argument 확인 (옵션)
    if 'args' in checkpoint:
        print("\n=== [3] Saved Arguments ===")
        print(checkpoint['args'])

except Exception as e:
    print(f"Error loading checkpoint: {e}")

Loading checkpoint from: /storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-BASELINE5/42/best.pt ...

=== [1] Top-level Keys ===
dict_keys(['model_state_dict', 'epoch', 'val_auc_mean', 'val_aucs_per_source', 'args'])


=== [2] Model Parameter Dimensions ===
basis_cls                                          | Shape: (1, 1, 768)
latent_graph.node_embeddings                       | Shape: (8, 8, 768)
latent_graph.q_proj.weight                         | Shape: (64, 768)
latent_graph.q_proj.bias                           | Shape: (64,)
latent_graph.k_proj.weight                         | Shape: (64, 768)
latent_graph.k_proj.bias                           | Shape: (64,)
gnn_experts.graph_gnns.0.linear.weight             | Shape: (192, 768)
gnn_experts.graph_gnns.0.linear.

In [7]:
import torch

ckpt1_path ="/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-GATFREEZE1/42/best.pt"
ckpt2_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-GATFREEZE2/42/best.pt"

# 로드
ckpt1 = torch.load(ckpt1_path, map_location='cpu')
ckpt2 = torch.load(ckpt2_path, map_location='cpu')

# state_dict 추출
state1 = ckpt1.get('model_state_dict', ckpt1.get('state_dict', ckpt1))
state2 = ckpt2.get('model_state_dict', ckpt2.get('state_dict', ckpt2))

# LCG node_embeddings 비교
key = "latent_graph.node_embeddings"

if key in state1 and key in state2:
    emb1 = state1[key]
    emb2 = state2[key]
    
    print(f"Shape: {emb1.shape}")
    print(f"Are they identical? {torch.allclose(emb1, emb2)}")
    
    if not torch.allclose(emb1, emb2):
        print(f"\nMax difference: {torch.max(torch.abs(emb1 - emb2)).item():.6e}")
        print(f"Mean difference: {torch.mean(torch.abs(emb1 - emb2)).item():.6e}")
else:
    print(f"'{key}' not found!")

Shape: torch.Size([8, 8, 768])
Are they identical? False

Max difference: 1.096577e-01
Mean difference: 5.116507e-03


In [13]:
import torch

ckpt1_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-log1/42/best.pt"


ckpt2_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-log2/42/best.pt"
ckpt1 = torch.load(ckpt1_path, map_location='cpu')
ckpt2 = torch.load(ckpt2_path, map_location='cpu')

state1 = ckpt1.get('model_state_dict', ckpt1.get('state_dict', ckpt1))
state2 = ckpt2.get('model_state_dict', ckpt2.get('state_dict', ckpt2))

key = "latent_graph.node_embeddings"

emb1 = state1[key]
emb2 = state2[key]

print(f"Shape: {emb1.shape}")
print(f"Are they identical? {torch.allclose(emb1, emb2)}")

if not torch.allclose(emb1, emb2):
    print(f"\nMax difference: {torch.max(torch.abs(emb1 - emb2)).item():.6e}")
    print(f"Mean difference: {torch.mean(torch.abs(emb1 - emb2)).item():.6e}")
else:
    print("\n✅ Centroids are IDENTICAL!")

Shape: torch.Size([8, 8, 768])
Are they identical? False

Max difference: 6.006138e-02
Mean difference: 5.362091e-03


In [12]:
import torch
import os

# ---------------------------------------------------------
# [설정] 비교할 두 체크포인트 경로 (Phase 1: best_vanilla.pt)
# ---------------------------------------------------------
#ckpt1_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-fixseedline1112_1/42/best_vanilla.pt"
ckpt1_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-20251208DATALOADERtest4_MKL1/42/best_vanilla.pt"
ckpt2_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-20251208DATALOADERtest5_MKL2/42/best_vanilla.pt"
def compare_checkpoints(path1, path2):
    print(f">>> Comparing Checkpoints:")
    print(f"    File 1: ...{path1[-50:]}")
    print(f"    File 2: ...{path2[-50:]}")

    if not os.path.exists(path1) or not os.path.exists(path2):
        print("❌ Error: One of the files does not exist.")
        return

    # CPU로 로드
    ckpt1 = torch.load(path1, map_location='cpu')
    ckpt2 = torch.load(path2, map_location='cpu')

    # state_dict 추출
    state1 = ckpt1.get('model_state_dict', ckpt1.get('state_dict', ckpt1))
    state2 = ckpt2.get('model_state_dict', ckpt2.get('state_dict', ckpt2))

    # 키 개수 확인
    keys1 = set(state1.keys())
    keys2 = set(state2.keys())

    if keys1 != keys2:
        print(f"❌ Mismatch in keys!")
        print(f"    Unique to File 1: {keys1 - keys2}")
        print(f"    Unique to File 2: {keys2 - keys1}")
        return

    print(f"✅ Key sets match. Comparing {len(keys1)} tensors...")

    all_match = True
    max_diff_global = 0.0

    for key in sorted(list(keys1)):
        p1 = state1[key]
        p2 = state2[key]

        # 텐서 타입/Shape 확인
        if p1.shape != p2.shape:
            print(f"❌ Shape Mismatch [{key}]: {p1.shape} vs {p2.shape}")
            all_match = False
            continue

        # Float 텐서만 값 비교 (LongTensor 등은 eq로 비교)
        if p1.is_floating_point():
            diff = torch.abs(p1 - p2)
            max_diff = torch.max(diff).item()
            if max_diff > 0: # 0이 아니면 차이가 있음
                max_diff_global = max(max_diff_global, max_diff)
                
                # 허용 오차 (e-7 정도는 부동소수점 연산 순서 차이로 발생 가능)
                if max_diff > 1e-6: 
                    print(f"❌ Value Mismatch [{key}]")
                    print(f"    Max Diff: {max_diff:.6e}")
                    print(f"    Mean Diff: {torch.mean(diff).item():.6e}")
                    all_match = False
        else:
            if not torch.equal(p1, p2):
                print(f"❌ Value Mismatch (Integer/Bool) [{key}]")
                all_match = False

    print("-" * 50)
    if all_match and max_diff_global == 0:
        print("🎉 PERFECT MATCH! (Phase 1 is perfectly deterministic)")
    elif all_match and max_diff_global < 1e-6:
        print(f"⚠️ Near Perfect Match. (Max Diff: {max_diff_global:.6e} <= 1e-6)")
        print("   -> This is likely due to GPU floating point associativity.")
        print("   -> Functionally considered Identical.")
    else:
        print(f"❌ FAILED. Phase 1 results are DIFFERENT. (Max Diff: {max_diff_global:.6e})")

if __name__ == "__main__":
    compare_checkpoints(ckpt1_path, ckpt2_path)

>>> Comparing Checkpoints:
    File 1: ...on-20251208DATALOADERtest4_MKL1/42/best_vanilla.pt
    File 2: ...on-20251208DATALOADERtest5_MKL2/42/best_vanilla.pt
✅ Key sets match. Comparing 176 tensors...
--------------------------------------------------
🎉 PERFECT MATCH! (Phase 1 is perfectly deterministic)


In [16]:
import torch
import os

# ==============================================================================
# [설정] 경로 지정 (사용자가 제공한 전체 경로 활용)
# ==============================================================================
# best.pt가 있는 정확한 폴더 경로 (42번 시드 폴더)
target_dir = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-v4_patience/42"

# 1. 가져올 체크포인트 파일 (target_dir 내의 best.pt)
source_ckpt_path = os.path.join(target_dir, "best.pt")

# 2. 저장될 파일 이름 (target_dir 내에 저장)
output_filename = "lcg_init_centroids_extracted.pt"
output_path = os.path.join(target_dir, output_filename)

# ==============================================================================
# [실행] 추출 및 저장 로직
# ==============================================================================
def extract_and_save_lcg_in_place(ckpt_path, save_path):
    print(f"🚀 Reading Checkpoint: {ckpt_path}")
    
    if not os.path.exists(ckpt_path):
        print(f"❌ Error: Checkpoint not found at -> {ckpt_path}")
        return

    try:
        # CPU로 로드
        ckpt = torch.load(ckpt_path, map_location='cpu')
        
        # State Dict 확보
        state_dict = ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt))
        
        # LCG Embedding 키 (key)
        target_key = "latent_graph.node_embeddings"
        
        if target_key in state_dict:
            centroids = state_dict[target_key]
            print(f"✅ LCG Embeddings Found! Shape: {centroids.shape}")
            
            # 저장할 데이터 구성
            save_data = {
                'node_embeddings': centroids,  # [M, K, D]
                'source_ckpt': ckpt_path,      # 출처 기록
                'description': "Extracted from 42/best.pt"
            }
            
            # 메타데이터 (args) 있으면 같이 저장
            if 'args' in ckpt:
                save_data['args'] = ckpt['args']

            # 파일 저장
            torch.save(save_data, save_path)
            
            print(f"\n💾 Saved Successfully to:")
            print(f"   -> {save_path}")
            print("\n[사용법] main.py 실행 시 다음 인자를 추가하세요:")
            print(f'   --transfer_lcg_path "{save_path}"')
            
        else:
            print(f"❌ Key '{target_key}' not found in state_dict.")
            
    except Exception as e:
        print(f"❌ Exception occurred: {e}")

# 함수 실행
extract_and_save_lcg_in_place(source_ckpt_path, output_path)

🚀 Reading Checkpoint: /storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-v4_patience/42/best.pt
✅ LCG Embeddings Found! Shape: torch.Size([8, 8, 768])

💾 Saved Successfully to:
   -> /storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-v4_patience/42/lcg_init_centroids_extracted.pt

[사용법] main.py 실행 시 다음 인자를 추가하세요:
   --transfer_lcg_path "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alph